# Week 1 exercise — Technical question explainer

Ask any technical question and get a streamed explanation from a **frontier** model
(`gpt-4o-mini`) and a **local** model (`llama3.2` via Ollama). One function drives both,
because Ollama exposes an OpenAI-compatible API.

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display

load_dotenv(override=True)

MODEL_GPT = "gpt-4o-mini"
MODEL_LLAMA = "llama3.2"

frontier = OpenAI()  # uses OPENAI_API_KEY
local = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")  # Ollama, OpenAI-compatible

SYSTEM = "You are a helpful technical tutor. Explain clearly and concisely, with a short example when useful. Use markdown."

In [2]:
def explain(question, client, model):
    """Stream an explanation and render it live as markdown."""
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": question},
    ]
    stream = client.chat.completions.create(model=model, messages=messages, stream=True)
    reply = ""
    handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        reply += chunk.choices[0].delta.content or ""
        update_display(Markdown(reply), display_id=handle.display_id)
    return reply

In [3]:
# Type over this to ask anything
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

### Frontier model — gpt-4o-mini

In [4]:
_ = explain(question, frontier, MODEL_GPT)

The code you've provided is a Python expression that uses a combination of a generator, a set comprehension, and the `yield from` statement. Here’s a breakdown of what each part does:

### Breakdown of the Code

1. **Set Comprehension**:
   ```python
   {book.get("author") for book in books if book.get("author")}
   ```
   This part creates a set of authors from a collection of `books`. Here's how it works:
   - `for book in books`: Iterates over each `book` in the `books` list.
   - `book.get("author")`: Retrieves the value associated with the key "author" from the `book` dictionary.
   - `if book.get("author")`: This condition checks if the "author" exists and is truthy (i.e., not `None`, empty string, etc.). If it is, the author is included in the set.
   - The entire expression returns a set of unique authors because sets automatically handle duplicate values.

2. **Yield from**:
   ```python
   yield from ...
   ```
   The `yield from` statement is used to yield all values from a generator or iterable. In this context, it is applied to the set comprehension created earlier, effectively allowing you to yield each unique author one at a time.

### Purpose of the Code

This code snippet would typically be part of a generator function, and its purpose is to return each unique author from the list of books one at a time. Since it uses `yield from`, it allows for efficient handling of the authors without the need to store them in a separate list or array.

### Example

Here’s a simple example demonstrating how this code might be used:

```python
def get_unique_authors(books):
    yield from {book.get("author") for book in books if book.get("author")}

# Sample data
books = [
    {"title": "Book 1", "author": "Author A"},
    {"title": "Book 2", "author": "Author B"},
    {"title": "Book 3", "author": "Author A"},  # Duplicate author
    {"title": "Book 4"}  # No author
]

# Using the generator
for author in get_unique_authors(books):
    print(author)
```

### Output
```
Author A
Author B
```

In this example, the generator function `get_unique_authors` yields unique authors from the `books` list, demonstrating the use of `yield from` with a set comprehension effectively.

### Local model — llama3.2 (Ollama)

In [5]:
_ = explain(question, local, MODEL_LLAMA)

**Generator Expression**
==========================

The given code uses a combination of two concepts: generator expression and `yield from`.

### Generator Expression

A generator expression is a compact way to create a generator, which is an iterable object that can be used to generate a sequence of values, one at a time.

Example:
```python
numbers = (x**2 for x in range(3))  # generator expression
print(next(numbers))  # prints 0
print(next(numbers))  # prints 1
print(next(numbers))  # prints 4
```

In this example, the generator expression creates a sequence of squares of numbers from `range(3)`.

### Yield from

`yield from` is a keyword used to pause the execution of a generator and delegate control to another generator. It allows you to yield values from another generator without having to implement a loop in Python code.

Example:
```python
def inner_generator():
    print("Inner gen")

    def small_gen():
        for i in [1, 2]:
            print(i)
    yield from small_gen()

inner_generator()
```

Output:
```
Inner gen
1
2
```

In this example, the `inner_generator` yields values from a nested generator called `small_gen`.

### Code Explanation

Now, let's apply these concepts to the original code:

```python
yield from {book.get("author") for book in books if book.get("author")}
```

This code uses a combination of `dict.get()` and a generator expression wrapped inside `{...}`. The dictionary values are obtained using `.get("author")`, and the result is an iterator.

The generator expression `(book.get("author")) for book in books if book.get("author")` yields values (i.e., authors) from each item `book` in the list `books`. However, due to a potential AttributeError when searching the dictionary value which may not exist, Python catches the error and uses `None` instead.

So, this code snippet essentially extracts all the author names from the `books` list while handling cases where authors might be missing.

Example Code with `List of Books**
```python
import json

class Book:
    def __init__(self, **kwargs):
        self.__dict__.update(kwargs)

books = [
    {"title": "Pride and Prejudice", "author":"Jane Austen"},
    {"title": "Wuthering Heights",},  # intentionally missing author field
    {"title": "Frankenstein", "author": "Mary Shelley"}
]

author_names = yield from {book.get("author") for book in books if book.get("author")}
print(author_names) # prints [None, None, 'Mary Shelley']
```